# Topic 5: Incremental Load Patterns

> **Phase 5 → Week 8 → Topic 5**

---

## Full Load vs Incremental Load

```
FULL LOAD (simple, expensive):
  Every run: delete everything, reload from scratch
  → Safe, always correct
  → Too slow for tables > a few GB
  → Wastes resources re-processing unchanged data

INCREMENTAL LOAD (fast, complex):
  Every run: process only NEW or CHANGED records since last run
  → Much faster for large tables
  → Requires tracking "what changed" (watermark / CDC)
  → More complex: must handle late arrivals, schema changes, duplicates

When to use each:
  Full load:        table < 1GB, simple daily batch, dimension tables
  Incremental:      large fact tables, hourly loads, streaming-like patterns
```

---

## Interview Cheat Sheet

**Q: What is a watermark in batch ETL?**
> A watermark is a saved timestamp (or ID) marking the last successfully processed record. On the next run, the pipeline reads only records where `modified_at > watermark`. After processing, it saves the new max `modified_at` as the new watermark. Watermarks are stored in an external state store (database table, config file, Delta log) and must be updated atomically with the write to avoid re-processing or missing data.

**Q: What is CDC (Change Data Capture)?**
> CDC captures every INSERT, UPDATE, and DELETE from a source database as a stream of change events. Instead of periodically scanning the full table, CDC reads the database transaction log. Spark can consume CDC events from Debezium (Kafka connector), DMS (AWS), or directly from Delta Lake's Change Data Feed. Each event has an `op` column: `I`=insert, `U`=update, `D`=delete.

**Q: What is an upsert (MERGE) and why is it important for incremental loads?**
> An upsert (UPDATE + INSERT) processes incoming records: if a record's key already exists in the target, UPDATE it; if it's new, INSERT it. In Spark, this requires Delta Lake's `MERGE INTO` (or `DeltaTable.merge()`). Without Delta, you'd need a full rewrite of the affected partition, which is expensive. MERGE is the cornerstone of incremental fact table loads and SCD (Slowly Changing Dimension) updates.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import json
import os

spark = SparkSession.builder \
    .appName("Week8 - Incremental Patterns") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Ready")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/16 08:15:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Ready


## Part 1: Watermark-Based Incremental Load

In [2]:
from datetime import datetime, timedelta

# Simulate a source table with a modified_at column
source_data = spark.createDataFrame([
    ("O001", 250.0, "delivered", datetime(2024, 1, 15, 10, 0)),
    ("O002",  89.5, "pending",   datetime(2024, 1, 16, 11, 0)),
    ("O003", 500.0, "delivered", datetime(2024, 1, 17, 12, 0)),
    ("O004",  45.0, "cancelled", datetime(2024, 1, 18, 9, 0)),
    ("O005", 175.0, "delivered", datetime(2024, 1, 19, 14, 0)),
], ["order_id", "amount", "status", "modified_at"])

WATERMARK_FILE = "/tmp/watermark.json"

def load_watermark():
    if os.path.exists(WATERMARK_FILE):
        with open(WATERMARK_FILE) as f:
            return datetime.fromisoformat(json.load(f)["last_run"])
    return datetime(2000, 1, 1)  # epoch start = first run

def save_watermark(ts: datetime):
    with open(WATERMARK_FILE, "w") as f:
        json.dump({"last_run": ts.isoformat()}, f)

def run_incremental_load():
    watermark = load_watermark()
    print(f"Running incremental load. Watermark: {watermark}")

    # Read only new/changed records
    new_records = source_data.filter(F.col("modified_at") > F.lit(watermark))
    batch_count = new_records.count()
    print(f"  Found {batch_count} new records")

    if batch_count > 0:
        new_records.show()
        # Process and write...
        new_records.write.mode("append").parquet("/tmp/incremental_output")

        # Update watermark to max modified_at in this batch
        new_watermark = new_records.agg(F.max("modified_at")).collect()[0][0]
        save_watermark(new_watermark)
        print(f"  Watermark updated to: {new_watermark}")
    else:
        print("  No new records — skipping")

# Run 1: fresh start, processes all records
print("=== Run 1 ===")
run_incremental_load()

=== Run 1 ===
Running incremental load. Watermark: 2000-01-01 00:00:00


  Found 5 new records


+--------+------+---------+-------------------+
|order_id|amount|   status|        modified_at|
+--------+------+---------+-------------------+
|    O001| 250.0|delivered|2024-01-15 10:00:00|
|    O002|  89.5|  pending|2024-01-16 11:00:00|
|    O003| 500.0|delivered|2024-01-17 12:00:00|
|    O004|  45.0|cancelled|2024-01-18 09:00:00|
|    O005| 175.0|delivered|2024-01-19 14:00:00|
+--------+------+---------+-------------------+



  Watermark updated to: 2024-01-19 14:00:00


In [3]:
# Run 2: watermark set, only new records (none here)
print("=== Run 2 (no new data) ===")
run_incremental_load()

# Simulate new records arriving
new_source_data = spark.createDataFrame([
    ("O006", 300.0, "delivered", datetime(2024, 1, 20, 8, 0)),
    ("O007", 120.0, "pending",   datetime(2024, 1, 21, 10, 0)),
], ["order_id", "amount", "status", "modified_at"])

source_data = source_data.union(new_source_data)  # simulate new rows in source

print("=== Run 3 (2 new records) ===")
run_incremental_load()

# Clean up
os.remove(WATERMARK_FILE)

=== Run 2 (no new data) ===
Running incremental load. Watermark: 2024-01-19 14:00:00


  Found 0 new records
  No new records — skipping
=== Run 3 (2 new records) ===
Running incremental load. Watermark: 2024-01-19 14:00:00


  Found 2 new records


+--------+------+---------+-------------------+
|order_id|amount|   status|        modified_at|
+--------+------+---------+-------------------+
|    O006| 300.0|delivered|2024-01-20 08:00:00|
|    O007| 120.0|  pending|2024-01-21 10:00:00|
+--------+------+---------+-------------------+



  Watermark updated to: 2024-01-21 10:00:00


## Part 2: Partition-Based Incremental (Date Partitions)

In [4]:
# Common pattern: source partitioned by date, load yesterday's partition daily

# Simulate partitioned source: /data/dt=2024-01-*/
for dt, data in [
    ("2024-01-15", [("O001", 250.0)]),
    ("2024-01-16", [("O002", 89.5)]),
    ("2024-01-17", [("O003", 500.0)]),
]:
    df = spark.createDataFrame(data, ["order_id", "amount"])
    df.write.mode("overwrite").parquet(f"/tmp/daily_source/dt={dt}")

# Daily incremental load: read only today's partition
def load_daily_partition(load_date: str):
    path = f"/tmp/daily_source/dt={load_date}"
    try:
        df = spark.read.parquet(path)
        print(f"Loaded {df.count()} rows for dt={load_date}")
        # Write to output table
        df.withColumn("dt", F.lit(load_date)) \
          .write.mode("overwrite").parquet(f"/tmp/daily_output/dt={load_date}")
    except Exception:
        print(f"No data for dt={load_date} — skipping")

# Simulate daily runs
for d in ["2024-01-15", "2024-01-16", "2024-01-17", "2024-01-18"]:
    load_daily_partition(d)

# Read the full output
all_loaded = spark.read.parquet("/tmp/daily_output")
print(f"\nTotal loaded across all dates: {all_loaded.count()} rows")
all_loaded.show()

Loaded 1 rows for dt=2024-01-15


Loaded 1 rows for dt=2024-01-16


Loaded 1 rows for dt=2024-01-17


No data for dt=2024-01-18 — skipping


26/05/16 08:16:05 WARN DataSource: [COLUMN_ALREADY_EXISTS] The column `dt` already exists. Consider to choose another name or rename the existing column.



Total loaded across all dates: 3 rows
+--------+------+----------+
|order_id|amount|        dt|
+--------+------+----------+
|    O001| 250.0|2024-01-15|
|    O003| 500.0|2024-01-17|
|    O002|  89.5|2024-01-16|
+--------+------+----------+



## Part 3: Upsert (MERGE) with Delta Lake

In [5]:
try:
    from delta.tables import DeltaTable
    DELTA_AVAILABLE = True
except ImportError:
    DELTA_AVAILABLE = False

print("Delta Lake MERGE/Upsert pattern (requires Delta Lake):")
print("  target.alias('t')")
print("    .merge(updates.alias('u'), 'u.order_id = t.order_id')")
print("    .whenMatchedUpdateAll()")
print("    .whenNotMatchedInsertAll()")
print("    .execute()")
print()
print(f"Delta Lake available: {DELTA_AVAILABLE}")
if not DELTA_AVAILABLE:
    print("Showing pattern as documentation only — Delta 3.0 incompatible with Spark 3.5")


Delta Lake MERGE/Upsert pattern (requires Delta Lake):
  target.alias('t')
    .merge(updates.alias('u'), 'u.order_id = t.order_id')
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()

Delta Lake available: True


In [6]:
# Parquet workaround for upsert (when Delta is not available)
def upsert_parquet(target_path, updates_df, key_cols):
    """
    Poor-man's upsert for plain Parquet.
    WARNING: not atomic — use Delta Lake in production!
    """
    try:
        existing = spark.read.parquet(target_path)
        # Remove existing rows for updated keys
        existing_minus_updated = existing.join(updates_df.select(key_cols), on=key_cols, how="left_anti")
        # Union and write
        merged = existing_minus_updated.union(updates_df)
    except Exception:
        merged = updates_df  # first write

    merged.write.mode("overwrite").parquet(target_path)

# Demo
initial = spark.createDataFrame([
    ("O001", 250.0, "pending"),
    ("O002", 89.5,  "delivered"),
], ["order_id", "amount", "status"])

initial.write.mode("overwrite").parquet("/tmp/parquet_upsert_demo")

updates = spark.createDataFrame([
    ("O001", 250.0, "delivered"),  # update: pending → delivered
    ("O003", 175.0, "pending"),    # new insert
], ["order_id", "amount", "status"])

upsert_parquet("/tmp/parquet_upsert_demo", updates, "order_id")

print("After Parquet upsert:")
spark.read.parquet("/tmp/parquet_upsert_demo").orderBy("order_id").show()

After Parquet upsert:


+--------+------+---------+
|order_id|amount|   status|
+--------+------+---------+
|    O001| 250.0|delivered|
|    O002|  89.5|delivered|
|    O003| 175.0|  pending|
+--------+------+---------+



## Part 4: Late-Arriving Data

In [7]:
print("""
Late-Arriving Data Problem:
════════════════════════════════════════════════════════════════
Scenario: Daily ETL runs at midnight for "yesterday's" data.
But some events from yesterday arrive 2 days late (network issues, retries).

Watermark approach:
  Problem: watermark = last_run_time. Late records have event_time < watermark
           → they look like "already processed" and get skipped!

Solutions:

Option 1: Re-process the last N days (overlap window)
  watermark = last_run - 3 days  # re-read 3 days, dedup on write
  → Simple, catches most late arrivals
  → Processes more data than necessary

Option 2: Partition overwrite with late window
  Daily partition job: overwrite partitions for last 7 days
  Late data for those days gets included automatically
  → Good for partition-based loads (dt=YYYY-MM-DD)

Option 3: Delta Lake MERGE with event_time filter
  Always merge on primary key regardless of watermark
  Delta ACID ensures no duplicates
  → Most robust, requires Delta Lake

Option 4: Structured Streaming with event-time watermark
  Spark Streaming has built-in late data handling with watermarks
  (covered in Week 10)
════════════════════════════════════════════════════════════════
""")

# Option 1 demo: overlap window
def load_with_overlap(batch_date: datetime, overlap_days: int = 3):
    start = batch_date - timedelta(days=overlap_days)
    print(f"Loading window: {start} to {batch_date} ({overlap_days}-day overlap)")
    # Then dedup on write

from datetime import datetime, timedelta
load_with_overlap(datetime(2024, 1, 20), overlap_days=3)


Late-Arriving Data Problem:
════════════════════════════════════════════════════════════════
Scenario: Daily ETL runs at midnight for "yesterday's" data.
But some events from yesterday arrive 2 days late (network issues, retries).

Watermark approach:
  Problem: watermark = last_run_time. Late records have event_time < watermark
           → they look like "already processed" and get skipped!

Solutions:

Option 1: Re-process the last N days (overlap window)
  watermark = last_run - 3 days  # re-read 3 days, dedup on write
  → Simple, catches most late arrivals
  → Processes more data than necessary

Option 2: Partition overwrite with late window
  Daily partition job: overwrite partitions for last 7 days
  Late data for those days gets included automatically
  → Good for partition-based loads (dt=YYYY-MM-DD)

Option 3: Delta Lake MERGE with event_time filter
  Always merge on primary key regardless of watermark
  Delta ACID ensures no duplicates
  → Most robust, requires Delta Lake



## Exercises

1. Write a full watermark-based incremental loader: reads from a source DataFrame filtered by `modified_at > watermark`, writes to Parquet, updates the watermark. Test it with 3 simulated runs.
2. What are the risks of storing a watermark in a local file (like `/tmp/watermark.json`)? Where should production watermarks be stored?
3. Write the Parquet upsert function and test it: (a) initial load, (b) update 2 rows, (c) insert 3 new rows. Verify the final state is correct.
4. An orders table is partitioned by `order_date`. New orders always arrive for today and yesterday (late arrivals). Design an incremental load strategy.
5. What is the "dual-write" problem in incremental loads? How does Delta Lake's MERGE solve it atomically?

In [8]:
# Answer Q2: Production watermark storage
print("""
Answer Q2: Watermark storage in production

Local file (/tmp/watermark.json) problems:
  ✗ Lost when the container/machine restarts
  ✗ Not shared across multiple workers
  ✗ Not versioned / no audit trail
  ✗ No atomic update guarantee

Production watermark storage options:
  ✓ Database table (Postgres, MySQL, RDS)
    UPDATE etl_watermarks SET last_run=NOW() WHERE job='orders_load'
    → Atomic, durable, queryable audit trail

  ✓ Delta Lake _delta_log
    Store watermark in table properties or a dedicated watermark Delta table
    → Co-located with data, versioned, time-travel

  ✓ Cloud parameter store (AWS SSM, GCP Secret Manager)
    → Managed, access-controlled, no database needed

  ✓ Airflow XCom / workflow metadata
    → If using Airflow: pass watermark between tasks as XCom

Answer Q5: Dual-write problem
  Without atomicity, an incremental load has TWO writes:
    1. Write new records to output
    2. Update watermark
  If the process crashes between 1 and 2:
    → Data was written but watermark NOT updated
    → Next run re-processes the same data → duplicates!
  If the process crashes before 1 but after 2:
    → Watermark updated but data NOT written
    → Data is lost!

  Delta MERGE solves this: the entire merge (update + insert + watermark)
  is one atomic transaction. Either ALL succeeds or ALL is rolled back.
""")


Answer Q2: Watermark storage in production

Local file (/tmp/watermark.json) problems:
  ✗ Lost when the container/machine restarts
  ✗ Not shared across multiple workers
  ✗ Not versioned / no audit trail
  ✗ No atomic update guarantee

Production watermark storage options:
  ✓ Database table (Postgres, MySQL, RDS)
    UPDATE etl_watermarks SET last_run=NOW() WHERE job='orders_load'
    → Atomic, durable, queryable audit trail

  ✓ Delta Lake _delta_log
    Store watermark in table properties or a dedicated watermark Delta table
    → Co-located with data, versioned, time-travel

  ✓ Cloud parameter store (AWS SSM, GCP Secret Manager)
    → Managed, access-controlled, no database needed

  ✓ Airflow XCom / workflow metadata
    → If using Airflow: pass watermark between tasks as XCom

Answer Q5: Dual-write problem
  Without atomicity, an incremental load has TWO writes:
    1. Write new records to output
    2. Update watermark
  If the process crashes between 1 and 2:
    → Data was